<a href = "https://www.pieriantraining.com"><img src="../PT Centered Purple.png"> </a>

<em style="text-align:center">Copyrighted by Pierian Training</em>

# Advanced Settings with AutoGluon

In this notebook we are going to inspect some advanced settings like smaller hyperparameter sets.

The same dataset as in the previous notebook will be used, however to save time for the demonstrations, we'll shrink the dataset to just 10% of the size.

In [1]:
from autogluon.tabular import TabularDataset, TabularPredictor
from rich.jupyter import display

In [2]:
data = TabularDataset("data/uber/uber.csv")
# To save time, let's make the data a little smaller!
data = data.sample((len(data)//3),random_state=42)

In [3]:
len(data)

66666

In [4]:
train_size = len(data)//10
seed = 42
train_data = data.sample(train_size, random_state=seed)
test_data = data.drop(train_data.index)

### Presets

Autogluon comes with many presets that you can pass to the training routine.
The training process will be adjusted depending on these presets.

You can find the documentation about all presets here:
https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.fit.html#:~:text=num_bag_sets%20is%20specified.-,presets
Currently the following presets are available:
[‘best_quality’, ‘high_quality’, ‘good_quality’, ‘medium_quality’, ‘optimize_for_deployment’, ‘interpretable’, ‘ignore_text’]


To get the best possible model you can use the *best_quality* preset - this will however drastically increase the training time.

To reduce the training time we can exclude both Neural Networks via <br />
**predictor.fit(train_data, presets=presets, excluded_model_types=["NN_TORCH", "FASTAI"])**



In [5]:
save_path = 'uber_predictors_preset'
presets = ["best_quality"]
predictor = TabularPredictor(label="fare_amount", path=save_path)
predictor.fit(train_data, presets=presets, excluded_model_types=["NN_TORCH", "FASTAI"])

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.15
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       6.77 GB / 32.00 GB (21.2%)
Disk Space Avail:   94.86 GB / 926.35 GB (10.2%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether t

Let's check the leaderboard to get a ranking of the individual models

In [2]:
leaderBoard_results = predictor.leaderboard()
display(leaderBoard_results)

NameError: name 'predictor' is not defined

In [1]:
def mostrar_leaderboard_optimizado(leaderboard_df, sort_by='balanced', tolerance_pct=0.05):
        """
        Filtra y ordena el leaderboard de AutoGluon según criterios de producción.

        Parámetros:
        -----------
        leaderboard_df : pd.DataFrame
            El DataFrame devuelto por predictor.leaderboard(...)
        sort_by : str, por defecto 'balanced'
            - 'score': Ordena de mejor a peor precisión.
            - 'speed': Ordena de más rápido a más lento en predecir (pred_time_val).
            - 'balanced': Filtra modelos dentro de la tolerancia de precisión del mejor,
                          y los ordena de más rápidos a más lentos.
        tolerance_pct : float, por defecto 0.05 (5%)
            La pérdida máxima de rendimiento aceptable a cambio de velocidad.
        """
        df = leaderboard_df.copy()

        # Detecta automáticamente si evaluar con test_data (score_test) o validación (score_val)
        score_col = 'score_test' if 'score_test' in df.columns and df['score_test'].notna().any() else 'score_val'

        print(f"🔍 Evaluando con la columna: '{score_col}'")

        if sort_by == 'score':
            df_result = df.sort_values(by=score_col, ascending=False)
            print("🏆 Modelos ordenados por precisión (Score máximo primero):")

        elif sort_by == 'speed':
            df_result = df.sort_values(by='pred_time_val', ascending=True)
            print("⚡ Modelos ordenados por velocidad de predicción (Más rápidos primero):")

        elif sort_by == 'balanced':
            # 1. Encontrar el mejor score
            best_score = df[score_col].max()

            # 2. Calcular el límite inferior aceptable (funciona con métricas positivas y negativas)
            lower_bound = best_score - abs(best_score) * tolerance_pct

            # 3. Filtrar modelos viables
            df_viable = df[df[score_col] >= lower_bound]

            # 4. Ordenar viables por velocidad
            df_result = df_viable.sort_values(by='pred_time_val', ascending=True)

            print(f"⚖️ Enfoque Balanceado para Producción:")
            print(f"   • Mejor score absoluto: {best_score:.4f}")
            print(f"   • Umbral aceptable (tolerancia del {tolerance_pct*100}%): {lower_bound:.4f}")
            print(f"   • Mostrando modelos sobre el umbral, ordenados de más rápidos a más lentos:")

        else:
            raise ValueError("sort_by debe ser 'score', 'speed' o 'balanced'")

        # Mostrar la tabla estilizada en el notebook
        display(df_result)
        return df_result

In [ ]:
best_balanced = mostrar_leaderboard_optimizado(
        leaderBoard_results,
        sort_by='balanced',
        tolerance_pct=0.05 # 5% de tolerancia
    )

In [10]:
y_test = test_data["fare_amount"]
test_data = test_data.drop(columns=["fare_amount"])

y_pred = predictor.predict(test_data)
metrics = predictor.evaluate_predictions(y_true=y_test, y_pred=y_pred, auxiliary_metrics=True)


Evaluation: root_mean_squared_error on test data: -5.948512879596984
	Note: Scores are always higher_is_better. This metric score can be multiplied by -1 to get the metric value.
Evaluations on test data:
{
    "root_mean_squared_error": -5.948512879596984,
    "mean_squared_error": -35.384805478731195,
    "mean_absolute_error": -2.5376794884308183,
    "r2": 0.6577656018723376,
    "pearsonr": 0.8113561769828805,
    "median_absolute_error": -1.5141555786132814
}


We can see, that the result is slightly superior compared to the previous notebook (where MAE was > 1.9)

Alternatively, we can pass a time_limit for the whole training.
Let's use a time limit of 1800 seconds

In [ ]:
save_path = 'uber_predictors_preset2'
presets = ["best_quality"]
predictor = TabularPredictor(label="fare_amount", path=save_path)
predictor.fit(train_data, presets=presets, time_limit=1800)

In [12]:

y_pred = predictor.predict(test_data)
metrics = predictor.evaluate_predictions(y_true=y_test, y_pred=y_pred, auxiliary_metrics=True)


Evaluation: root_mean_squared_error on test data: -5.948512879596984
	Note: Scores are always higher_is_better. This metric score can be multiplied by -1 to get the metric value.
Evaluations on test data:
{
    "root_mean_squared_error": -5.948512879596984,
    "mean_squared_error": -35.384805478731195,
    "mean_absolute_error": -2.5376794884308183,
    "r2": 0.6577656018723376,
    "pearsonr": 0.8113561769828805,
    "median_absolute_error": -1.5141555786132814
}


### Hyperparameter presets

To increase the training speed, we can select between different hyperparameter presets:

https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.fit.html#:~:text=was%20also%20specified.-,hyperparameters

Currently, the following sets are available: [‘default’, ‘light’, ‘very_light’, ‘toy’, ‘multimodal’]

In [ ]:
save_path = 'uber_predictors_hyperparams_very_light'

predictor = TabularPredictor(label="fare_amount", path=save_path)
predictor.fit(train_data, presets=presets, time_limit=1800, hyperparameters="very_light")

In [ ]:

y_pred = predictor.predict(test_data)
metrics = predictor.evaluate_predictions(y_true=y_test, y_pred=y_pred, auxiliary_metrics=True)


## Deployment
After identification of the best model we might want to deploy it.

To do so let us at first create a clone of the original model, which we will then further postprocess

In [ ]:
save_path = 'uber_predictors_preset'  # This was the best model from above
save_path_clone = save_path + '_clone'

predictor = TabularPredictor.load(save_path)
path_clone = predictor.clone(path=save_path_clone)


Optimize inference speed.
Next we can call the **refit_full()** function which retrains the model and optimizes it for inference time (https://auto.gluon.ai/dev/api/autogluon.tabular.TabularPredictor.refit_full.html) if bagging was used.

In [ ]:
predictor.refit_full()

Last but not least, we can use **clone_for_deployment()** which only keeps the data required for inference and thus requires less storage

In [ ]:
predictor.clone_for_deployment("uber_predictor_deployment")

## Feature Engineering
Autogluon comes with a heavily optimized automated feature engineering pipeline, the **AutoMLPipelineFeatureGenerator** https://auto.gluon.ai/stable/api/autogluon.features.html

A deatailed overview over the implemented routines can be found here: https://auto.gluon.ai/stable/tutorials/tabular/tabular-feature-engineering.html

In [ ]:
from autogluon.features import AutoMLPipelineFeatureGenerator

We have already seen in the previous lecture, that autogluon succesfully recognized the date in the timestamp string and converted it to multiple columns. All of this functionality is implemented in the above described generator.
It can be used as follows (if you are familiar with sklearn, you already know this routine):

In [ ]:
train_data.head()

In [ ]:
feature_pipeline = AutoMLPipelineFeatureGenerator()
transformed_features = feature_pipeline.fit_transform(train_data)

In [ ]:
transformed_features.head()

### Custom feature pipeline
In case you want to define your own feature preprocessing pipeline you can use the **PipelineFeatureGenerator()**

In [ ]:
from autogluon.features import PipelineFeatureGenerator, DatetimeFeatureGenerator

In [ ]:
feature_gen_custom = PipelineFeatureGenerator(generators=[
    DatetimeFeatureGenerator(features=["year", "month", "day", "hour", "minute"]),
    
])

In [ ]:
transformed_custom = feature_gen_custom.fit_transform(train_data)

In [ ]:
transformed_custom.head()

# Great Job!

In [ ]:
print('done')